In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2005_2006"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ACQ_D.xpt"
csv_file = DATA_DIR / "ACQ_D_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched copy for fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2005-2006 ACQ_D XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. STRUCTURE CHECK
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ACD010A",
    "ACD010B",
    "ACD010C",
    "ACD040"
]


if len(df) != 7267:
    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 5:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "ACQ_D columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: ACQ_D structure = "
    "7,267 rows x 5 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-VALUE ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- TINY VALUE CHECK ---")

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
    )

    raise ValueError(
        "STOP: ACQ_D unexpectedly contains "
        "tiny XPORT values. "
        "Do NOT automatically replace them."
    )


print(
    "PASS: No tiny-value artifact exists."
)


# ============================================================
# 5. TRUE ZERO CHECK
# ============================================================

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


zero_total = int(
    zero_counts.sum()
)


print(
    "\nTrue numeric zeros:",
    zero_total
)


if zero_total != 0:

    print(
        zero_counts[
            zero_counts > 0
        ]
    )

    raise ValueError(
        "Unexpected zero values found. "
        "Review before proceeding."
    )


print(
    "PASS: Source contains no numeric zero values."
)


# ============================================================
# 6. CONFIRM ALL VALUES ARE WHOLE-NUMBER CODES
# ============================================================

print("\n--- FRACTIONAL VALUE CHECK ---")


fraction_failures = []


for col in df.columns:

    values = (
        df[col]
        .dropna()
    )


    if values.empty:
        continue


    fractional = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    count = int(
        fractional.sum()
    )


    print(
        f"{col:<10}: "
        f"fractional values = {count:,}"
    )


    if count != 0:

        fraction_failures.append(
            col
        )


if fraction_failures:

    raise ValueError(
        "STOP: Genuine decimal values found in:\n"
        + str(fraction_failures)
    )


print(
    "PASS: All ACQ_D nonmissing values "
    "are exact whole-number identifiers/codes."
)


# ============================================================
# 7. SEQN VALIDATION
# ============================================================

print("\n--- SEQN ---")


print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(df["SEQN"].isna().sum()) != 0
    or
    int(df["SEQN"].duplicated().sum()) != 0
    or
    df["SEQN"].min() != 31129
    or
    df["SEQN"].max() != 41474
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 8. CDC FREQUENCY CHECK - ACD010A
# ============================================================

EXPECTED_ACD010A = {
    1.0: 5164,
    77.0: 6
}


actual = (
    df["ACD010A"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_ACD010A
    or
    int(df["ACD010A"].isna().sum()) != 2097
):

    raise ValueError(
        "ACD010A differs from CDC.\n"
        f"Actual: {actual}"
    )


print(
    "\nPASS: ACD010A matches CDC."
)


# ============================================================
# 9. CDC FREQUENCY CHECK - ACD010B
# ============================================================

EXPECTED_ACD010B = {
    8.0: 24
}


actual = (
    df["ACD010B"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_ACD010B
    or
    int(df["ACD010B"].isna().sum()) != 7243
):

    raise ValueError(
        "ACD010B differs from CDC.\n"
        f"Actual: {actual}"
    )


print(
    "PASS: ACD010B matches CDC."
)


# ============================================================
# 10. CDC FREQUENCY CHECK - ACD010C
# ============================================================

EXPECTED_ACD010C = {
    9.0: 240
}


actual = (
    df["ACD010C"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_ACD010C
    or
    int(df["ACD010C"].isna().sum()) != 7027
):

    raise ValueError(
        "ACD010C differs from CDC.\n"
        f"Actual: {actual}"
    )


print(
    "PASS: ACD010C matches CDC."
)


# ============================================================
# 11. CDC FREQUENCY CHECK - ACD040
# ============================================================

EXPECTED_ACD040 = {
    1.0: 658,
    2.0: 242,
    3.0: 399,
    4.0: 284,
    5.0: 377,
    7.0: 1,
    9.0: 1
}


actual = (
    df["ACD040"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_ACD040
    or
    int(df["ACD040"].isna().sum()) != 5305
):

    raise ValueError(
        "ACD040 differs from CDC.\n"
        f"Actual: {actual}"
    )


print(
    "PASS: ACD040 matches CDC."
)


# ============================================================
# 12. KEEP EXPECTED SOURCE FOR FIDELITY
# ============================================================

#
# ACQ_D requires NO value correction.
#
# Expected source therefore equals original XPT.
#

expected_df = source_df.copy()


# ============================================================
# 13. FORMAT ALL RELEASED VARIABLES AS WHOLE NUMBERS
# ============================================================

#
# We already proved all nonmissing values are exact
# whole numbers.
#
# Therefore:
#
# SEQN:
# 31129.0 -> 31129
#
# ACD010A:
# 1.0 -> 1
#
# ACD040:
# 5.0 -> 5
#
# Missing:
# NaN -> blank
#

for col in df.columns:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: SEQN and all questionnaire fields "
    "formatted without unnecessary .0."
)


# ============================================================
# 14. VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


value_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    if not np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        value_failures.append(
            col
        )


if value_failures:

    raise ValueError(
        "Unexpected value changes in:\n"
        + str(value_failures)
    )


print(
    "PASS: All 5 variables retain "
    "their original XPT values."
)


# ============================================================
# 15. EXPORT CSV
# ============================================================

#
# Missing SAS numeric values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 16. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 17. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


numeric_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        numeric_failures.append(
            col
        )


if numeric_failures:

    raise ValueError(
        "XPT -> CSV numeric differences found:\n"
        + str(numeric_failures)
    )


print(
    "PASS: All 5 CSV columns numerically "
    "match original ACQ_D XPT."
)


# ============================================================
# 18. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print(
    "\nMissing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 19. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    reader = csv.DictReader(f)
    raw_rows = list(reader)


# ============================================================
# 20. STRICT RAW FORMAT CHECK
# ============================================================

#
# Since every released ACQ_D variable is a whole-number
# identifier/code field, each raw token should be:
#
# str(int(original XPT value))
#
# Example:
#
# 31129.0 -> "31129"
# 1.0     -> "1"
# 77.0    -> "77"
# NaN     -> ""
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in expected_df.columns:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    (
                        i + 2,
                        col,
                        expected_text,
                        actual_text
                    )
                )


print(
    "\nRaw-format mismatches:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "Examples:",
        raw_examples
    )

    raise ValueError(
        "ACQ_D raw formatting differs."
    )


print(
    "PASS: Raw CSV formatting matches expected "
    "whole-number questionnaire representation."
)


# ============================================================
# 21. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "\nValues ending '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        "Examples:",
        dot_zero_examples
    )

    raise ValueError(
        "Unnecessary .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 22. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = (
            value.lower()
        )


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        "Examples:",
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 23. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("ACQ_D CONVERSION COMPLETE")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "Tiny-value corrections:",
    0
)

print(
    "Genuine decimal variables:",
    0
)

print(
    "SEQN/code formatting:",
    "integer"
)

print(
    "XPT -> CSV numeric differences:",
    "0 expected"
)

print(
    "Missing-position differences:",
    "0 expected"
)

print(
    "Raw-format differences:",
    "0 expected"
)

print(
    "Values ending .0:",
    "0 expected"
)

NHANES 2005-2006 ACQ_D XPT -> CSV
Rows: 7,267
Columns: 5
PASS: ACQ_D structure = 7,267 rows x 5 columns.

--- TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: No tiny-value artifact exists.

True numeric zeros: 0
PASS: Source contains no numeric zero values.

--- FRACTIONAL VALUE CHECK ---
SEQN      : fractional values = 0
ACD010A   : fractional values = 0
ACD010B   : fractional values = 0
ACD010C   : fractional values = 0
ACD040    : fractional values = 0
PASS: All ACQ_D nonmissing values are exact whole-number identifiers/codes.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 31129.0
Maximum: 41474.0
PASS: SEQN validated.

PASS: ACD010A matches CDC.
PASS: ACD010B matches CDC.
PASS: ACD010C matches CDC.
PASS: ACD040 matches CDC.

PASS: SEQN and all questionnaire fields formatted without unnecessary .0.

--- VALUE FIDELITY BEFORE EXPORT ---
PASS: All 5 variables retain their original XPT values.

CSV created:
C:\Users\User\Desktop\Projects\Clinical_Neighbourhood_ML_Data_Predicto